# ITM 454 - Natural Language Processing
## Language Models and N-grams

This notebook will guide you through creating simple language models using unigrams, bigrams, and n-grams. You'll learn to calculate probabilities and predict the next word in a sequence.

---

In [1]:
import nltk
from nltk.corpus import brown
import random

### 1. Choose Text Corpus

In [2]:
print(len(brown.sents(categories='news')))
# print(brown.categories()) # uncomment this to see the list of the categories

# Get sentences from Brown corpus (first 1000 for faster processing)
sentences = brown.sents(categories='news')[:1000]
print(f"Number of sentences: {len(sentences)}")
print(f"First sentence: {sentences[1]}")

4623
Number of sentences: 1000
First sentence: ['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.']


### 2. Pre-processing
- **Normalize**: convert to lowercase, keep only alphabets.
- Add sentence markers (needed for Text Generation).

In [3]:
# Convert to lowercase and add sentence markers
processed_sentences = []
for sentence in sentences:
    # Normalizing by converting to lowercase and add start/end markers
    # Note that we don't need to remove stopwords here to keep the sentence structure
    processed = ['<START>'] + [word.lower() for word in sentence if word.isalpha()] + ['<END>']
    processed_sentences.append(processed)

print(f"\nProcessed first sentence: {processed_sentences[0]}")


Processed first sentence: ['<START>', 'the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', '<END>']


### 3. Build Unigram Model
Count the frequency of each word in the corpus.

In [4]:
# Flatten all sentences into one list of words
all_words = []
for sentence in processed_sentences:
    all_words.extend(sentence) # why extend? = cuz it could take an itarable

print(f"Total words in corpus: {len(all_words)}")

# Count word frequencies (unigrams)
unigram_counts = nltk.FreqDist(all_words)
total_words = len(all_words)

print(f"Vocabulary size: {len(unigram_counts)}")
print(f"\nMost common unigrams:")
for word, count in unigram_counts.most_common(10):
    probability = count / total_words
    print(f"{word}: {count} ({probability:.4f})")

Total words in corpus: 20848
Vocabulary size: 3804

Most common unigrams:
the: 1569 (0.0753)
<START>: 1000 (0.0480)
<END>: 1000 (0.0480)
of: 676 (0.0324)
to: 549 (0.0263)
a: 440 (0.0211)
in: 438 (0.0210)
and: 377 (0.0181)
for: 266 (0.0128)
that: 211 (0.0101)


In [5]:
# Function to calculate unigram probability
def unigram_probability(word):
    return unigram_counts[word] / total_words

# Test the function
test_words = ['the', 'house', 'pizza', 'hello', 'world']
print("Unigram probabilities:")
for word in test_words:
    prob = unigram_probability(word)
    print(f"P({word}) = {prob:.6f}")

Unigram probabilities:
P(the) = 0.075259
P(house) = 0.001679
P(pizza) = 0.000000
P(hello) = 0.000000
P(world) = 0.000240


### 4. Build Bigram Model
Count pairs of consecutive words and calculate conditional probabilities.

In [6]:
# Generate bigrams from all sentences
bigrams = []
for sentence in processed_sentences:
    for i in range(len(sentence) - 1):
        bigram = (sentence[i], sentence[i + 1])
        bigrams.append(bigram)

print(f"Total bigrams: {len(bigrams)}")
print(f"First 10 bigrams: {bigrams[:10]}")

# Count bigram frequencies
bigram_counts = nltk.FreqDist(bigrams)
print(f"\nMost common bigrams:")
for bigram, count in bigram_counts.most_common(10):
    print(f"{bigram}: {count}")

Total bigrams: 19848
First 10 bigrams: [('<START>', 'the'), ('the', 'fulton'), ('fulton', 'county'), ('county', 'grand'), ('grand', 'jury'), ('jury', 'said'), ('said', 'friday'), ('friday', 'an'), ('an', 'investigation'), ('investigation', 'of')]

Most common bigrams:
('of', 'the'): 220
('<START>', 'the'): 207
('in', 'the'): 153
('to', 'the'): 71
('on', 'the'): 68
('for', 'the'): 67
('<START>', 'he'): 60
('that', 'the'): 46
('at', 'the'): 40
('the', 'state'): 40


In [7]:
# Function to calculate bigram probability P(word2|word1)
def bigram_probability(word1, word2):
    bigram_count = bigram_counts[(word1, word2)]
    word1_count = unigram_counts[word1]
    
    if word1_count == 0:
        return 0
    
    return bigram_count / word1_count

# Test bigram probabilities
test_bigrams = [('the', 'house'), ('in', 'the'), ('<START>', 'the'), ('fruit', 'juice')]
print("Bigram probabilities:")
for word1, word2 in test_bigrams:
    prob = bigram_probability(word1, word2)
    print(f"P({word2}|{word1}) = {prob:.6f}")

Bigram probabilities:
P(house|the) = 0.009560
P(the|in) = 0.349315
P(the|<START>) = 0.207000
P(juice|fruit) = 0.000000


### 5. NLP Task 1 - Text Generation
Use our language models to generate text by predicting the next word.

In [12]:
# Generate text using bigram model
def generate_text_bigram(start_word='<START>', max_length=20):
    result = [start_word]
    current_word = start_word
    
    for _ in range(max_length):
        # Find all words that can follow the current word
        possible_words = []
        for (w1, w2), count in bigram_counts.items():
            if w1 == current_word:
                possible_words.extend([w2] * count)  # Add word multiple times based on frequency
        
        if not possible_words:
            break
        
        # Choose next word randomly based on probability
        next_word = random.choice(possible_words)
        result.append(next_word)
        
        if next_word == '<END>':
            break
            
        current_word = next_word
    
    return ' '.join(result[1:-1])  # Remove START and END markers

# Generate some sample text
print("Generated text using bigram model:")
for i in range(5):
    generated = generate_text_bigram()
    print(f"{i+1}. {generated}")

print('---')
print("will: will", generate_text_bigram("will")) # generate next word after will
print("what is the controversy about? : ", generate_text_bigram("about")) # generate next word after will

Generated text using bigram model:
1. and revised to wage the city expects to the correct method and form two chambers as the first believed
2. the last night in the ministers has studied law
3. hood flies over methods courses
4. georgia voters in the hughes said has served notice was offered there was a dispute with the final legislative
5. however the monthly cost of one time mitchell said that they will lack of the latter two years old
---
will: will succeed in each
what is the controversy about? :  new taxes


### 6. NLP Task 2 - Next Word Prediction
Given a context, predict the most likely next word based on frequency.

In [9]:
def next_word_prediction(word):
    possible_words = []
    for (w1, w2), count in bigram_counts.items():
        if w1 == word:
            possible_words.append((w2,count))  # Add word multiple times based on frequency
    
    next_possible_words = sorted(possible_words, key=lambda x: x[1], reverse=True)[:5] # sorted and chose only the top 5
    return next_possible_words

# Test next word prediction
test_contexts = ['the', 'will', 'in']
print("Next word predictions using bigram model:")
for context in test_contexts:
    predictions = next_word_prediction(context)
    print(f"After '{context}': {predictions}")

Next word predictions using bigram model:
After 'the': [('state', 40), ('city', 29), ('united', 25), ('administration', 18), ('first', 18)]
After 'will': [('be', 33), ('take', 4), ('not', 3), ('go', 2), ('ask', 2)]
After 'in': [('the', 153), ('a', 19), ('his', 10), ('an', 8), ('<END>', 7)]


In [10]:
# Next word prediction based on highest probabilities
curr = "it"
sent = curr
while len(next_word_prediction(curr)) > 0:
    sent += " " + next_word_prediction(curr)[0][0]
    curr = next_word_prediction(curr)[0][0]

print(sent)

it was a year <END>


### Homework
Create a new notebook or make a copy of this notebook.
1. Try another categories of Brown Corpus
2. Try changing from first 1000 to first 5000 or randomly at any points (min 1000)
3. Try removing stopwords in Normalization steps
4. Observe the differences in text generation and next word prediction
5. **Go to Canvas:** Upload your notebook, and write your observation in ***N-gram Languge Model exercise***